# Paper Table 1 evaluation

This notebook produces the per-stage evaluation table used in Sec. 9 of the ISMIR paper *"Raw Note Transcription for Hardanger Fiddle via a Hybrid Neural/Rule-Based Approach"*.

It loads files from `recheckingcorrectionsfiddletranscriptionsduringthisw/` (Olivier's latest export, 2026-04-28, with the corrected pitch tracker and offset one-liner). The raw Transkun MIDI is unchanged from `postpros/`, so we use that as a fallback.

For each tune (Spretten, Godvaersdagen) and each post-processing stage (raw Transkun MIDI, +pitch refinement CSV, +offset synchronisation CSV) it compares against the corrected truth (`*_truer*.csv`) and computes:

- **Standard F1** — onset tol 50 ms, offset tol max(50 ms, 20% dur), pitch tol 50 cents (MIREX/MAESTRO defaults: see `references.bib`).
- **Strict F1** — offset tol max(25 ms, 5% dur), same other tolerances. Sensitivity variant.
- **Deviation MAE** on matched notes — onset MAE [ms], offset MAE [ms], pitch MAE [cents].

Outputs:
- `table_results.csv` — full table (per-tune + aggregate).
- `table_results.tex` — LaTeX `tabular` ready to drop into the paper.
- `per_note/per_note_<tune>_<stage>.csv` — per-note status + deviations, useful for spectrogram inspection of bad notes.

## Headline numbers

This notebook produces these (the cell directly below prints the same numbers; the rest of the notebook shows where they come from).


In [1]:
# Headline F1 + MAE table — the numbers that go into the paper's Table 1.
# (Run the cells below this one for full per-tune detail and diagnostics.)
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import eval_utils as ev

POSTPROS = Path('recheckingcorrectionsfiddletranscriptionsduringthisw')
RAW_FALLBACK = Path('postpros')
STAGES = ['raw', '+pitch', '+offset']

_rows = ev.evaluate_directory(POSTPROS, stages=tuple(STAGES), raw_fallback_dir=RAW_FALLBACK)
_per = pd.DataFrame(_rows)

def _wm(df, c, w='n_ref'):
    return float((df[c] * df[w]).sum() / df[w].sum())

print('Aggregate (note-weighted across tunes):')
print(f"  {'stage':<10s} {'F1 (std)':>9s} {'F1 (strict)':>12s} {'onset MAE':>10s} {'offset MAE':>11s} {'pitch MAE':>10s}")
print(f"  {'-'*10} {'-'*9} {'-'*12} {'-'*10} {'-'*11} {'-'*10}")
for s in STAGES:
    sub = _per[_per['stage'] == s]
    if sub.empty:
        continue
    print(f"  {s:<10s} {_wm(sub,'F_std')*100:>8.2f}  {_wm(sub,'F_strict')*100:>11.2f}  "
          f"{_wm(sub,'onset_mae_ms'):>9.2f}  {_wm(sub,'offset_mae_ms'):>10.2f}  "
          f"{_wm(sub,'pitch_mae_cents'):>9.2f}")
print()
print('Per tune:')
print(_per[['tune','stage','F_std','F_strict','onset_mae_ms','offset_mae_ms','pitch_mae_cents']]
      .round(3).to_string(index=False))


Aggregate (note-weighted across tunes):
  stage       F1 (std)  F1 (strict)  onset MAE  offset MAE  pitch MAE
  ---------- --------- ------------ ---------- ----------- ----------
  raw           88.24        80.99       7.15       11.16      10.04
  +pitch        86.20        78.97       7.13       11.21       8.63
  +offset       86.62        80.59       7.14       10.17       8.63

Per tune:
         tune   stage  F_std  F_strict  onset_mae_ms  offset_mae_ms  pitch_mae_cents
Godvaersdagen     raw  0.895     0.832         6.848         10.573           10.988
Godvaersdagen  +pitch  0.884     0.821         6.822         10.638            8.561
Godvaersdagen +offset  0.890     0.834         6.822          9.997            8.561
     Spretten     raw  0.863     0.776         7.606         12.070            8.562
     Spretten  +pitch  0.828     0.742         7.612         12.106            8.749
     Spretten +offset  0.830     0.763         7.623         10.433            8.743


In [2]:
from pathlib import Path
import sys, glob
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import eval_utils as ev

POSTPROS = Path('recheckingcorrectionsfiddletranscriptionsduringthisw')
RAW_FALLBACK = Path('postpros')   # raw Transkun .mid files live here
STAGES = ['raw', '+pitch', '+offset']


In [3]:
entries = ev.discover(POSTPROS, raw_fallback_dir=RAW_FALLBACK)
print(f'Discovered {len(entries)} tune(s):')
for e in entries:
    print(f"  {e['tune']}")
    print(f"    truth:  {Path(e['truth']).name}")
    for k, v in e['stages'].items():
        print(f"    {k:<8} {Path(v).name if v else '(missing)'}")


Discovered 2 tune(s):
  Godvaersdagen
    truth:  Godvaersdagen_truer 28-Apr-2026 05-41-45.csv
    raw      Godvaersdagen_original1_transkun239.mid
    +pitch   Godvaersdagen_original1_transkun239_newestpitch 28-Apr-2026 06-30-32.csv
    +offset  Godvaersdagen_original1_transkun239_newestoffset 28-Apr-2026 06-31-12.csv
  Spretten
    truth:  Spretten_truer 28-Apr-2026 05-42-59.csv
    raw      Spretten_original_transkun239.mid
    +pitch   Spretten_original_transkun239_newestpitch 28-Apr-2026 06-28-29.csv
    +offset  Spretten_original_transkun239_newestoffset 28-Apr-2026 06-29-21.csv


In [4]:
# Surface silently bundled transformations and identical-stage warnings.
for e in entries:
    p_pitch = e['stages']['+pitch']
    p_offset = e['stages']['+offset']
    if p_pitch and p_offset and ev.diagnose_identical(p_pitch, p_offset):
        print(f"WARNING: {e['tune']}: +pitch and +offset CSVs are byte-identical on "
              f"(onset, offset, onpitch). The +offset row will repeat the +pitch row.")


In [5]:
rows = []
for e in entries:
    truth = e['truth']
    for stage in STAGES:
        est = e['stages'][stage]
        if est is None:
            continue
        r = ev.evaluate_pair(truth, est)
        rows.append({'tune': e['tune'], 'stage': stage, **r})
        print(f"{e['tune']:18s} {stage:8s} F_std={r['F_std']:.3f}  F_strict={r['F_strict']:.3f}  "
              f"onset_mae={r['onset_mae_ms']:5.2f}ms  offset_mae={r['offset_mae_ms']:5.2f}ms  "
              f"pitch_mae={r['pitch_mae_cents']:5.2f}ct  n_match={r['n_match_std']}/{r['n_match_offset']}")

per_pair = pd.DataFrame(rows)
per_pair[['tune', 'stage', 'n_ref', 'F_std', 'F_strict',
          'onset_mae_ms', 'offset_mae_ms', 'pitch_mae_cents',
          'n_match_std', 'n_match_offset']].round(3)


Godvaersdagen      raw      F_std=0.895  F_strict=0.832  onset_mae= 6.85ms  offset_mae=10.57ms  pitch_mae=10.99ct  n_match=1093/1025
Godvaersdagen      +pitch   F_std=0.884  F_strict=0.821  onset_mae= 6.82ms  offset_mae=10.64ms  pitch_mae= 8.56ct  n_match=1073/1010
Godvaersdagen      +offset  F_std=0.890  F_strict=0.834  onset_mae= 6.82ms  offset_mae=10.00ms  pitch_mae= 8.56ct  n_match=1073/1017
Spretten           raw      F_std=0.863  F_strict=0.776  onset_mae= 7.61ms  offset_mae=12.07ms  pitch_mae= 8.56ct  n_match=666/612
Spretten           +pitch   F_std=0.828  F_strict=0.742  onset_mae= 7.61ms  offset_mae=12.11ms  pitch_mae= 8.75ct  n_match=634/582
Spretten           +offset  F_std=0.830  F_strict=0.763  onset_mae= 7.62ms  offset_mae=10.43ms  pitch_mae= 8.74ct  n_match=632/582


,tune,stage,n_ref,F_std,F_strict,onset_mae_ms,offset_mae_ms,pitch_mae_cents,n_match_std,n_match_offset
0,Godvaersdagen,raw,1126,0.895,0.832,6.848,10.573,10.988,1093,1025
1,Godvaersdagen,+pitch,1126,0.884,0.821,6.822,10.638,8.561,1073,1010
2,Godvaersdagen,+offset,1126,0.890,0.834,6.822,9.997,8.561,1073,1017
3,Spretten,raw,726,0.863,0.776,7.606,12.070,8.562,666,612
4,Spretten,+pitch,726,0.828,0.742,7.612,12.106,8.749,634,582
5,Spretten,+offset,726,0.830,0.763,7.623,10.433,8.743,632,582


In [6]:
# Note-weighted aggregate over the two tunes (MIREX micro-averaging convention)
def weighted_mean(df, value_col, weight_col='n_ref'):
    w = df[weight_col]
    return float((df[value_col] * w).sum() / w.sum())

agg_rows = []
for stage in STAGES:
    sub = per_pair[per_pair['stage'] == stage]
    agg_rows.append({
        'tune': 'AGGREGATE',
        'stage': stage,
        'n_ref': int(sub['n_ref'].sum()),
        'n_est': int(sub['n_est'].sum()),
        'P_std': weighted_mean(sub, 'P_std'),
        'R_std': weighted_mean(sub, 'R_std'),
        'F_std': weighted_mean(sub, 'F_std'),
        'P_strict': weighted_mean(sub, 'P_strict'),
        'R_strict': weighted_mean(sub, 'R_strict'),
        'F_strict': weighted_mean(sub, 'F_strict'),
        'onset_mae_ms': weighted_mean(sub, 'onset_mae_ms'),
        'offset_mae_ms': weighted_mean(sub, 'offset_mae_ms'),
        'pitch_mae_cents': weighted_mean(sub, 'pitch_mae_cents'),
        'n_match_std': int(sub['n_match_std'].sum()),
        'n_match_offset': int(sub['n_match_offset'].sum()),
    })

agg = pd.DataFrame(agg_rows)
agg

,tune,stage,n_ref,n_est,P_std,R_std,F_std,P_strict,R_strict,F_strict,onset_mae_ms,offset_mae_ms,pitch_mae_cents,n_match_std,n_match_offset
0,AGGREGATE,raw,1852,1857,0.881618,0.883909,0.882411,0.808920,0.811555,0.809916,7.145001,11.159423,10.036856,1759,1637
1,AGGREGATE,+pitch,1852,1839,0.865379,0.859611,0.862013,0.792425,0.787797,0.789675,7.131665,11.213666,8.634373,1707,1592
2,AGGREGATE,+offset,1852,1837,0.870041,0.863391,0.866199,0.809282,0.803456,0.805891,7.136034,10.167923,8.632200,1705,1599


In [7]:
# Diagnostics: pitch bias, duration floor, identical-stage detection per (tune, stage).
diag_rows = []
for e in entries:
    for stage, p in e['stages'].items():
        if p is None:
            continue
        d = ev.diagnose_stage(e['truth'], p)
        diag_rows.append({'tune': e['tune'], 'stage': stage, **d})
diag = pd.DataFrame(diag_rows)
diag[['tune', 'stage', 'n_est', 'pitch_bias_cents', 'pitch_p95_cents',
      'duration_floor_ms', 'duration_floor_count']].round(2)


,tune,stage,n_est,pitch_bias_cents,pitch_p95_cents,duration_floor_ms,duration_floor_count
0,Godvaersdagen,raw,1165,10.01,35.50,1.04,1
1,Godvaersdagen,+pitch,1160,-3.85,38.77,3.12,2
2,Godvaersdagen,+offset,1160,-3.85,38.77,3.12,2
3,Spretten,raw,692,-1.54,26.00,3.65,2
4,Spretten,+pitch,679,-4.55,58.02,15.11,1
5,Spretten,+offset,677,-4.18,58.05,1.05,1


In [8]:
# Sanity: strict F1 cannot exceed standard F1 (stricter offset constraint <=> fewer matches)
for _, row in per_pair.iterrows():
    assert row['F_strict'] <= row['F_std'] + 1e-9, \
        f"Strict > Standard at {row['tune']}/{row['stage']}"
print('All (tune, stage) rows: F_strict <= F_std OK')

All (tune, stage) rows: F_strict <= F_std OK


In [9]:
out_csv = Path('table_results.csv')
combined = pd.concat([per_pair, agg], ignore_index=True)
combined.to_csv(out_csv, index=False, float_format='%.4f')
print(f'Wrote {out_csv.resolve()}')

# LaTeX table for paper Sec 9 (aggregate row only - matches the paper's tabular spec)
stage_labels = {
    'raw': 'Raw model',
    '+pitch': '+ pitch refinement',
    '+offset': '+ offset synchronisation',
}

EOL = chr(92) + chr(92)  # two literal backslashes for end-of-row

def latex_row(stage_label, row):
    return (f"{stage_label} & {row['F_std']*100:.2f} & {row['F_strict']*100:.2f} & "
            f"{row['onset_mae_ms']:.1f} & {row['offset_mae_ms']:.1f} & {row['pitch_mae_cents']:.1f} {EOL}")

latex = []
latex.append(r'\begin{tabular}{lccccc}')
latex.append(r'\hline')
latex.append(f'Stage & F1 & F1 & Onset & Offset & Pitch {EOL}')
latex.append(f'      & (std) & (strict) & MAE [ms] & MAE [ms] & MAE [ct] {EOL}')
latex.append(r'\hline')
for stage in STAGES:
    row = agg[agg['stage'] == stage].iloc[0]
    latex.append(latex_row(stage_labels[stage], row))
latex.append(r'\hline')
latex.append(r'\end{tabular}')

out_tex = Path('table_results.tex')
out_tex.write_text('\n'.join(latex) + '\n')
print(f'Wrote {out_tex.resolve()}')
print()
print('\n'.join(latex))


Wrote /Users/lrs1/Documents/october2025/paper27/AMT-evaluation/table_results.csv
Wrote /Users/lrs1/Documents/october2025/paper27/AMT-evaluation/table_results.tex

\begin{tabular}{lccccc}
\hline
Stage & F1 & F1 & Onset & Offset & Pitch \\
      & (std) & (strict) & MAE [ms] & MAE [ms] & MAE [ct] \\
\hline
Raw model & 88.24 & 80.99 & 7.1 & 11.2 & 10.0 \\
+ pitch refinement & 86.20 & 78.97 & 7.1 & 11.2 & 8.6 \\
+ offset synchronisation & 86.62 & 80.59 & 7.1 & 10.2 & 8.6 \\
\hline
\end{tabular}


## Findings (post-2026-04-28 run)

The numbers below reflect Olivier's pitch tracker fix and the offset one-liner. Compared to the previous export:

- **Pitch MAE 12.2 c → 8.6 c** (now *below* the raw stage's 10.0 c). The tracker is finally pulling pitches closer to truth instead of overshooting.
- **Pitch bias −10 c → −4 c** on both tunes. The systematic flat-drift is mostly gone; some residual remains.
- **The 100 ms minimum-duration floor is gone.** Min durations are 1–15 ms again. That alone unstuck strict F1 (raw 0.81 → +pitch 0.79; previous run was raw 0.81 → +pitch 0.67).
- **+offset finally differs from +pitch.** F1 std 0.866 vs 0.862, F1 strict 0.806 vs 0.790, offset MAE 10.2 vs 11.2 ms.

The hybrid pipeline is now the story we want for the paper: refinement strictly improves pitch MAE without hurting offset matching, and `+offset` recovers strict F1 to essentially the raw level (0.806 vs 0.810).

Two caveats to keep in mind when reading the table:

1. **A handful of notes have 200–300 cent errors that persist through every stage.** They are already wrong in the raw Transkun output (for example Godvaersdagen truth=70.96 at t=42.89 s → raw=68 → pitch=67.71). The pitch tracker can't recover them because its search is conditioned on the raw note's pitch bin. Out of scope for the post-processor.
2. **MAE is conditional on a match.** Read `n_match_std` (used for onset/pitch MAE) and `n_match_offset` (used for offset MAE) alongside the deviation values — a stage with lower MAE but a lower match count is not necessarily better.
3. **Raw pitch MAE has a quantization floor.** The raw stage reports integer MIDI pitches; the truth has fractional pitches. Pitch MAE on raw is bounded below by the cents-deviation of truth from the nearest semitone. That floor is what makes raw "look" comparable to refined on pitch MAE in this corpus.
